# NB11 - Ablation study
Which design choices matter, and where does the model fail? Each variant uses the *same* outer splits (first `ablation.n_repeats` repeats) and is compared
with the main model by paired participant-level bootstrap.

| variant | question |
|---|---|
| no_augmentation / no_class_weighting | do these training choices help? |
| cnn2d_slices | 2D slices vs 3D volume |
| resnet10_scratch / resnet10_medicalnet | does pre-training help? (medicalnet needs `paths.medicalnet_weights`) |
| **mask_only_input** | shortcut probe: only the brain silhouette, no intensities. If this is about as good as the full model, size/shape (or skull-stripping edges) carries the signal |
| **eroded_boundary_input** | shortcut probe: outer brain shell removed. A big drop = reliance on the brain boundary |

Run time: roughly 1/3 of NB05 per variant.

In [ ]:
# ── 1. Environment Detection (Google Colab vs Local Machine) ──────────────
import sys, os, subprocess

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ── 2. Configure Paths ───────────────────────────────────────────────────
if IN_COLAB:
    # Google Colab paths (on your Google Drive)
    PROJECT  = '/content/drive/MyDrive/oasis_project'      # this unzipped project folder on Drive
    DATA_DIR = '/content/drive/MyDrive/OASIS1'             # folder with the 12 OASIS-1 disc archives (+ clinical csv)
    OUT_DIR  = '/content/drive/MyDrive/OASIS1_outputs'     # results are saved here (survives Colab disconnects)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{PROJECT}/requirements.txt'], check=False)
else:
    # Local machine paths (runs directly on your laptop)
    PROJECT  = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
    DATA_DIR = os.path.join(PROJECT, 'Dataset')
    OUT_DIR  = os.path.join(PROJECT, 'OASIS1_outputs')

# ── 3. install + load ─────────────────────────────────────────────────────
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)
from oasis_ad.config import load_config, Paths
from oasis_ad import pipeline as PL, DISCLAIMER
cfg = load_config(os.path.join(PROJECT, 'configs', 'default.yaml'),
                  {'paths.data_dir': DATA_DIR, 'paths.out_dir': OUT_DIR})
paths = Paths(cfg)
import torch, pandas as pd
print(DISCLAIMER)
print('Environment:', 'Google Colab' if IN_COLAB else 'Local Laptop')
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (on Colab switch to Runtime > Change runtime type > T4 GPU)')

In [ ]:
tab = PL.step10_ablations(cfg, paths, 'cnn3d')   # variants from cfg.ablation.variants
tab

Interpretation: a variant whose AUROC difference interval includes 0 does **not** differ detectably from the full model - report that honestly.